In [32]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import loadtxt
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import SGDClassifier


In [2]:
dataset_druggable = pd.read_csv("input/combined_DepMap_21Q3_druggable.csv")

In [3]:
dataset_ccle = pd.read_csv("input/combined_DepMap_21Q3_CCLE_expression.csv")

/users/ysu13/repos/drug_sensitivity_mlpred/output/oversample


In [29]:
dataset.iloc[1]

cell_line_name                                     786O
SHOC2                                          -0.21108
NDUFA12                                       -0.062542
SDAD1                                           -0.5537
FAM98A                                         -0.12355
                                                 ...   
BRD-K99506538-001-03-8::2.5::MTS004                   1
BRD-K99616396-001-05-1::2.499991421::MTS004           0
BRD-K99879819-001-02-1::2.5187366::MTS004             0
BRD-K99919177-001-01-3::2.5::MTS004                   0
BRD-M63173034-001-03-6::2.64076472::MTS004            1
Name: 1, Length: 22338, dtype: object

In [31]:
dataset = pd.read_csv("input/combined_DepMap_21Q3.csv")
num_gene = 17651
X = dataset.iloc[:, 1:num_gene+1]
drug_y_all = dataset.iloc[:, -4686:]
drug_list = drug_y_all.columns.tolist()

In [6]:
directory_path = "output/oversample"
os.makedirs(directory_path, exist_ok=True)

In [13]:
ROS = RandomOverSampler(random_state=72)
X_ovs, y_ovs = ROS.fit_resample(X, drug_y_all['BRD-A00100033-001-08-9::2.5::HTS'])

In [16]:
dataset.shape

(924, 22338)

In [7]:
a = XGBClassifier

In [11]:
{XGBClassifier:'XGBClassifier',
                   RandomForestClassifier:'RandomForestClassifier'
                   }[a]

'XGBClassifier'

In [36]:
def skl_drug_model(drug, model = XGBClassifier, oversample = True, model_params={}):

     assert model in [XGBClassifier, RandomForestClassifier, SGDClassifier], f' {model} type not supported'

     model_name = {
          XGBClassifier:'XGBClassifier',
          RandomForestClassifier:'RandomForestClassifier',
          SGDClassifier:'SGDClassifier'

      }[model]



     y = drug_y_all[drug]

     

     
     # split X and y into training and testing sets
     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)


     if oversample:
          ros = RandomOverSampler(random_state=72)
          X_res, y_res = ros.fit_resample(X_train, y_train)
          oversample = 'oversample'
     else:
          X_res, y_res = X_train, y_train
          oversample = ''

     
     
     # grid_search_parameters = {
        #"n_estimators": [100, 150, 250],
       # "max_depth" : [20, 50, 100, 200],
        # }



     grid_search_parameters = {
         "l1_ratio": [0.15, 0.1, 0.05],
         }
     # instantiate the classifier
     

    # k-fold cross validation using multiple metric evaluation
     kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
     #cv_results = cross_validate(xgbc, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1'], n_jobs = 10)
     if oversample:
          imba_pipeline = Pipeline([('sampling', RandomOverSampler(random_state=72)), 
                              ('classifier', model(n_jobs=20, **model_params))])
          grid_search_parameters = {'classifier__' + key: grid_search_parameters[key] for key in grid_search_parameters}
     else:
          imba_pipeline = model( n_jobs=20)
     #cross_val_score(imba_pipeline, X_train, y_train, scoring='recall', cv=kf)
     grid_imba = GridSearchCV(imba_pipeline, param_grid=grid_search_parameters, cv=kfold, scoring='average_precision_score',
                        return_train_score=True)
     grid_imba.fit(X_train, y_train) 

     best_params = {key.removeprefix('classifier__'):grid_imba.best_params_[key] for key in grid_imba.best_params_}
     print(best_params)
     model0 = model(**model_params, n_jobs=20)
     model0.set_params(**best_params)
     imba_pipeline = Pipeline([('sampling', RandomOverSampler(random_state=72)), 
                              ('classifier', model0)])
     cv_results = cross_validate(imba_pipeline, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs = 20)
     cv_results = pd.DataFrame(cv_results)
     cv_results.to_csv(f'output/{oversample}/{model_name}_cv_results_{drug}.csv')


     # declare parameters
     

     # fit the classifier to the training data
     model0.fit(X_res, y_res)

     # save the trained model
     joblib.dump(model0, f'output/{oversample}/{model_name}_{drug}.joblib')

    # make predictions on test data
     y_pred = model0.predict(X_test)

     print(drug,
         f'{model_name}_model_parameters', model0, "\n",
         "confusion_matrix:", "\n", confusion_matrix(y_test, y_pred), "\n",
         file=open(f'output/{oversample}/{model_name}_confusion_matrix.txt', "a"))

     model_report = classification_report(y_test, y_pred, output_dict=True, labels=np.unique(y_pred))
     model_report = pd.DataFrame(model_report).transpose()
    
     if (model_report.index == "1").any() == True:
          r1 = pd.DataFrame(model_report.loc["1"]).transpose()
          r1.to_csv(f'output/{oversample}/{model_name}_classification_report_{drug}.csv')
     else:
          print(drug, "Nothing predicted as 1",
               file=open(f'output/{oversample}/{model_name}_classification_report_log.txt', "a"))

     

     # feature importance with XGBoost
     if model in [XGBClassifier, RandomForestClassifier]:
          fi = pd.DataFrame({'feature': list(X_train.columns),
                    'importances': model0.feature_importances_ * 100}).\
                     sort_values('importances', ascending = False)
          fi.to_csv(f'output/{oversample}/{model_name}_feature_importance_{drug}.csv')
     return grid_imba

In [25]:
grid_search.cv_results_

{'mean_fit_time': array([2.40790286, 2.54538484, 2.89265499, 2.38947196, 2.54764538,
        2.92974715, 2.44389091, 2.49653816, 2.8900691 , 2.35250878,
        2.59743299, 2.88341689]),
 'std_fit_time': array([0.12274417, 0.12299602, 0.12952892, 0.11078133, 0.1321442 ,
        0.10698331, 0.11328307, 0.03785639, 0.04081947, 0.09719499,
        0.12912627, 0.03065442]),
 'mean_score_time': array([0.09776516, 0.09592943, 0.10795693, 0.0944366 , 0.09472213,
        0.11095028, 0.09322195, 0.09411416, 0.1069345 , 0.09539418,
        0.1030107 , 0.11045604]),
 'std_score_time': array([0.00104097, 0.00194667, 0.00096526, 0.00084937, 0.00084046,
        0.00623106, 0.00067869, 0.00220336, 0.00095855, 0.00221828,
        0.00383497, 0.00147739]),
 'param_classifier__max_depth': masked_array(data=[20, 20, 20, 50, 50, 50, 100, 100, 100, 200, 200, 200],
              mask=[False, False, False, False, False, False, False, False,
                    False, False, False, False],
        fill_value=

In [ ]:
def xgbc(drug, oversample = True):
     y = drug_y_all[drug]

     

     
     # split X and y into training and testing sets
     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)


     if oversample:
          ros = RandomOverSampler(random_state=72)
          X_res, y_res = ros.fit_resample(X_train, y_train)
          oversample = 'oversample'
     else:
          X_res, y_res = X_train, y_train
          oversample = ''

     # declare parameters
     parameters = {
         "n_estimators": 150,
         "eta" : 0.1,
         "gamma" : 0.0,
         "max_depth" : 100,
         "eval_metric": "auc"
         }

     
     grid_search_parameters = {
         "n_estimators": [100, 150, 250],
         "eta" : [0.1],
         "gamma" : [0.0],
         "max_depth" : [20, 50, 100, 200],
         }

     # instantiate the classifier
     xgb0 = XGBClassifier(**parameters, use_label_encoder=False, n_jobs=10)

     # fit the classifier to the training data
     xgb0.fit(X_res, y_res)

     # save the trained model
     joblib.dump(xgb0, "output/"+oversample+"/XGBoost_%s.joblib" % drug)

    # make predictions on test data
     y_pred = xgb0.predict(X_test)

     print(drug,
         "XGBoost_model_parameters", xgb0, "\n",
         "confusion_matrix:", "\n", confusion_matrix(y_test, y_pred), "\n",
         file=open("output/"+oversample+"/confusion_matrix.txt", "a"))

     model_report = classification_report(y_test, y_pred, output_dict=True, labels=np.unique(y_pred))
     model_report = pd.DataFrame(model_report).transpose()
    
     if (model_report.index == "1").any() == True:
          r1 = pd.DataFrame(model_report.loc["1"]).transpose()
          r1.to_csv("output/"+oversample+"/classification_report_%s.csv" % drug)
     else:
          print(drug, "Nothing predicted as 1",
               file=open("output/"+oversample+"/classification_report_log.txt", "a"))

    # k-fold cross validation using multiple metric evaluation
     kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
     #cv_results = cross_validate(xgbc, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1'], n_jobs = 10)
     if oversample:
          imba_pipeline = Pipeline([('sampling', SMOTE(random_state=72)), 
                              ('classifier', XGBClassifier( n_jobs=10))])
          grid_search_parameters = {'classifier__' + key: grid_search_parameters[key] for key in grid_search_parameters}
     else:
          imba_pipeline = XGBClassifier( n_jobs=10)
     #cross_val_score(imba_pipeline, X_train, y_train, scoring='recall', cv=kf)
     grid_imba = GridSearchCV(imba_pipeline, param_grid=grid_search_parameters, cv=kfold, scoring='f1',
                        return_train_score=True)
     grid_imba.fit(X_train, y_train) 
     
     cv_results = cross_validate(imba_pipeline, X_train, y_train, cv=kfold, scoring= ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs = 10)
     cv_results = pd.DataFrame(cv_results)
     cv_results.to_csv("output/"+oversample+"/cv_results_%s.csv" % drug)

     # feature importance with XGBoost
     fi = pd.DataFrame({'feature': list(X_train.columns),
               'importances': xgb0.feature_importances_ * 100}).\
                sort_values('importances', ascending = False)
     fi.to_csv("output/"+oversample+"/feature_importance_%s.csv" % drug)
     return grid_imba

In [24]:
starttime = time.time()
grid_search = skl_drug_model('BRD-A00100033-001-08-9::2.5::HTS',   oversample = True, model = RandomForestClassifier
)
endtime = time.time()

{'max_depth': 200, 'n_estimators': 100}


In [57]:
test_pipeline = make_pipeline(SMOTE(random_state=72), 
                              XGBClassifier( n_jobs=10))

In [74]:
grid_search.best_params_

{'classifier__eta': 0.1,
 'classifier__gamma': 0.0,
 'classifier__max_depth': 20,
 'classifier__n_estimators': 200}

In [26]:
pd.DataFrame(grid_search.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__max_depth,param_classifier__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,2.407903,0.122744,0.097765,0.001041,20,100,"{'classifier__max_depth': 20, 'classifier__n_e...",0.105263,0.195122,0.100000,...,0.143235,0.035870,2,1.0,1.0,1.0,1.0,1.0,1.0,0.0
1,2.545385,0.122996,0.095929,0.001947,20,150,"{'classifier__max_depth': 20, 'classifier__n_e...",0.054054,0.153846,0.054054,...,0.105591,0.045569,9,1.0,1.0,1.0,1.0,1.0,1.0,0.0
2,2.892655,0.129529,0.107957,0.000965,20,250,"{'classifier__max_depth': 20, 'classifier__n_e...",0.105263,0.153846,0.054054,...,0.135237,0.051446,3,1.0,1.0,1.0,1.0,1.0,1.0,0.0
3,2.389472,0.110781,0.094437,0.000849,50,100,"{'classifier__max_depth': 50, 'classifier__n_e...",0.054054,0.105263,0.105263,...,0.116074,0.038931,5,1.0,1.0,1.0,1.0,1.0,1.0,0.0
4,2.547645,0.132144,0.094722,0.000840,50,150,"{'classifier__max_depth': 50, 'classifier__n_e...",0.153846,0.200000,0.153846,...,0.134271,0.048923,4,1.0,1.0,1.0,1.0,1.0,1.0,0.0
5,2.929747,0.106983,0.110950,0.006231,50,250,"{'classifier__max_depth': 50, 'classifier__n_e...",0.105263,0.105263,0.102564,...,0.115819,0.021111,6,1.0,1.0,1.0,1.0,1.0,1.0,0.0
6,2.443891,0.113283,0.093222,0.000679,100,100,"{'classifier__max_depth': 100, 'classifier__n_...",0.105263,0.054054,0.105263,...,0.095590,0.020797,10,1.0,1.0,1.0,1.0,1.0,1.0,0.0
7,2.496538,0.037856,0.094114,0.002203,100,150,"{'classifier__max_depth': 100, 'classifier__n_...",0.105263,0.153846,0.105263,...,0.115565,0.037605,7,1.0,1.0,1.0,1.0,1.0,1.0,0.0
8,2.890069,0.040819,0.106934,0.000959,100,250,"{'classifier__max_depth': 100, 'classifier__n_...",0.054054,0.105263,0.054054,...,0.064296,0.039854,12,1.0,1.0,1.0,1.0,1.0,1.0,0.0
9,2.352509,0.097195,0.095394,0.002218,200,100,"{'classifier__max_depth': 200, 'classifier__n_...",0.105263,0.153846,0.105263,...,0.145479,0.037461,1,1.0,1.0,1.0,1.0,1.0,1.0,0.0


In [50]:
starttime = time.time()
xgbc('BRD-A00100033-001-08-9::2.5::HTS'
)
endtime = time.time()

/users/ysu13/miniforge3/envs/drug_sensitivity_ml/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [12:37:34] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1745056857893/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/users/ysu13/miniforge3/envs/drug_sensitivity_ml/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/users/ysu13/miniforge3/envs/drug_sen